# 🗣️ Natural Language Processing (NLP) — Practice Notebook
## Clear Concepts, Step by Step

---

**Welcome!**

Every day you type messages on WhatsApp, search on Google, or talk to Siri. Have you ever wondered — **how does the computer understand what you typed?**

That is what **Natural Language Processing (NLP)** is all about.

In this notebook you will:

| # | What you will learn |
|---|---------------------|
| 1 | What NLP is (and why it matters) |
| 2 | Why human language is hard for computers |
| 3 | The NLP workflow — the big picture |
| 4 | Clean and prepare text (preprocessing) |
| 5 | Turn words into numbers (vectorization) |
| 6 | Train a simple sentiment classifier |
| 7 | Make predictions on new text |

> 📖 **After this notebook**, open `nlp_workbook.ipynb` for a real-world project using Kenyan product reviews!


---
## 📦 Import the Tools

Before we start, we need to bring in the Python libraries we will use.
Think of this like getting your pens, ruler, and calculator out before a maths lesson.

In [1]:
# ── Text processing tools
import re                                         # For cleaning text (removing punctuation etc.)
import string                                     # Has a list of punctuation characters

# ── Data tools ─────
import pandas as pd                               # For working with tables

# ── Machine Learning (scikit-learn) ──────
from sklearn.feature_extraction.text import CountVectorizer   # Bag-of-Words
from sklearn.model_selection import train_test_split           # Split data
from sklearn.naive_bayes import MultinomialNB                  # Our classifier
from sklearn.metrics import accuracy_score, classification_report

print('✅ All tools loaded successfully!')

✅ All tools loaded successfully!


---
---
## 🧠 CONCEPT 1 — What is Natural Language Processing (NLP)?

### The Simple Definition

> **NLP** is a branch of Artificial Intelligence that teaches computers to **read, understand, and generate human language** — just like you do!

---

### A Kenyan Example 🇰🇪

Imagine **Amina** in Mombasa sends a WhatsApp message:

> *"M-Pesa ilifanya kazi vizuri leo, nimependa sana!"*  *(M-Pesa worked well today, I loved it!)*

Or **Wanjiru** in Nairobi tweets:

> *"Safaricom data bundles are too expensive, I'm switching to Airtel!"*

A company like Safaricom would love to automatically read **millions** of such messages and know:
- Is this customer **happy** or **unhappy**?
- What are people **complaining** about most?
- Which product needs to be **improved**?

Doing this by hand would take years. NLP does it in **seconds**.

---

### Where is NLP used today?

| Application | Example you know |
|-------------|------------------|
| **Chatbots** | Safaricom's Zuri assistant |
| **Search engines** | Google understands your question |
| **Translation** | Google Translate (English ↔ Kiswahili) |
| **Voice assistants** | Siri, Alexa, Google Assistant |
| **Spam detection** | Gmail moves junk mail to spam |
| **Sentiment analysis** | Companies reading Twitter feedback |

---
---
## ⚠️ CONCEPT 2 — Why is Human Language Hard for Computers?

Computers are very good at maths. But language is **messy and full of tricks**.

Here are the main reasons language is difficult:

---

### 2a. Ambiguity — The Same Words Mean Different Things

Read this sentence:
> *"Omondi spotted a man with a telescope."*

**Two possible meanings:**
- Omondi **used a telescope** to see the man.
- Omondi saw a man **who was carrying a telescope**.

A human knows from context. A computer gets confused!

---

### 2b. Context Changes Everything

> *"Safaricom is **killing** it with the new offers!"*

Here **"killing"** is a **positive** thing — it means doing very well.
But if a computer sees the word "killing" it might think this is a **negative** or dangerous sentence.

---

### 2c. Sarcasm

> *"Oh great, the KBS bus is late again. I'm SO surprised."* 🙄

A human knows this is **negative**. The word "great" and "surprised" would confuse a computer into thinking it is **positive**.

---

### 2d. Slang, Typos, and Sheng

> *"Hii game ni poa sana bana 🔥🔥"*  *(This game is very cool man)*

Computers trained on formal English have no idea what **"poa"** or **"bana"** means.

---

### 2e. Same Spelling, Different Meaning

> - I went to the **bank** to withdraw money. (a financial institution)
> - We sat by the river **bank**. (edge of a river)

---

### Summary

| Challenge | Example |
|-----------|---------|
| Ambiguity | "spotted a man with a telescope" |
| Context | "killing it" = doing well |
| Sarcasm | "SO surprised" = not surprised at all |
| Slang/Sheng | "poa", "bana", "manze" |
| Same word, different meaning | "bank" |

> 📌 This is why NLP is a whole field of study — language is far more complex than numbers!

---
---
## 🗺️ CONCEPT 3 — The NLP Workflow (The Big Picture)

Every NLP project follows the same general steps:

```
 Step 1️⃣  Collect Text Data          e.g. customer reviews, tweets
     ↓
 Step 2️⃣  Label the Data              e.g. tag each review as positive / negative
     ↓
 Step 3️⃣  Preprocess the Text         clean it: lowercase, remove punctuation, remove noise
     ↓
 Step 4️⃣  Vectorize (Text → Numbers)  turn words into numbers the model understands
     ↓
 Step 5️⃣  Train a Model               feed numbers + labels into a Machine Learning model
     ↓
 Step 6️⃣  Make Predictions            give the model new text and get a prediction
     ↓
 Step 7️⃣  Evaluate                    check how accurate the model is
```

> 📌 We will go through **every single step** in this notebook!

---
---
## 🧹 STEP 1 & 2 — Collect and Label Data

**Wanjiku** works at a data startup in Westlands, Nairobi. Her team collected social media comments about various Kenyan brands and **labeled** each one as positive or negative.

Here is a small sample of what they collected:

In [2]:
# ── Our sample labeled dataset ────────────────────────────────────────────────
# Each tuple = (text, label)
# Labels: 'positive' = happy customer,  'negative' = unhappy customer

data = [
    # Positive examples
    ("Safaricom customer care was so helpful and friendly!",         "positive"),
    ("M-Pesa is the best! I sent money to Kisumu in seconds.",       "positive"),
    ("The nyama choma at Carnivore Restaurant is absolutely amazing!","positive"),
    ("KCB bank app works perfectly, very smooth experience.",         "positive"),
    ("Nairobi Expressway saved me so much time today, love it!",      "positive"),
    ("Java House coffee is the best in Nairobi, highly recommend.",   "positive"),
    ("Equity Bank loan process was fast and stress-free.",            "positive"),
    ("Jambo Jet flights are affordable and the staff are kind.",      "positive"),
    ("The new Uhuru Park renovation is beautiful, well done!",        "positive"),
    ("Jumia delivery was super fast, got my package same day!",       "positive"),

    # Negative examples
    ("Safaricom data bundles are too expensive, very disappointed.",  "negative"),
    ("M-Pesa transaction failed three times, terrible service!",      "negative"),
    ("Nairobi traffic is a nightmare, spent 3 hours on Thika Road.",  "negative"),
    ("KCB ATM was out of cash again, very frustrating.",              "negative"),
    ("KBS bus was so late today, I missed my interview.",             "negative"),
    ("The food at that restaurant was cold and tasteless, awful.",    "negative"),
    ("Jumia delivered the wrong item, customer care was useless.",    "negative"),
    ("Airtel network kept dropping during my meeting, so annoying!",  "negative"),
    ("Long queues at Huduma Centre, wasted my entire morning.",       "negative"),
    ("The matatu conductor was rude and overcharged us.",             "negative"),
]

# Put the data into a pandas DataFrame (like a spreadsheet table)
df = pd.DataFrame(data, columns=['text', 'label'])

print('📊 Dataset shape:', df.shape)   # (rows, columns)
print()
print('Label counts:')
print(df['label'].value_counts())
print()
df.head(6)    # Show first 6 rows

📊 Dataset shape: (20, 2)

Label counts:
label
positive    10
negative    10
Name: count, dtype: int64



,text,label
0,Safaricom customer care was so helpful and fri...,positive
1,M-Pesa is the best! I sent money to Kisumu in ...,positive
2,The nyama choma at Carnivore Restaurant is abs...,positive
3,"KCB bank app works perfectly, very smooth expe...",positive
4,Nairobi Expressway saved me so much time today...,positive
5,"Java House coffee is the best in Nairobi, high...",positive


---
---
## 🧹 STEP 3 — Preprocess the Text

Raw text is **messy**. Before we feed it to a model, we need to clean it.

### What preprocessing does:

| Step | Before | After |
|------|--------|-------|
| **Lowercase** | `"Safaricom is GREAT!"` | `"safaricom is great!"` |
| **Remove punctuation** | `"safaricom is great!"` | `"safaricom is great"` |
| **Remove stopwords** | `"safaricom is great"` | `"safaricom great"` |

> 📌 **Stopwords** are very common words like *"is", "the", "a", "and"* that don't carry much meaning. Removing them helps the model focus on the important words.

Let's build a simple preprocessing function!

In [3]:
# ── Define our stopwords (common words we want to ignore) ─────────────────────
# We keep this simple — in a real project you would use the NLTK library's list
STOPWORDS = {
    'i', 'me', 'my', 'the', 'a', 'an', 'is', 'it', 'was', 'are', 'were',
    'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'shall', 'should', 'may', 'might', 'can', 'could',
    'not', 'no', 'nor', 'so', 'yet', 'both', 'and', 'but', 'or', 'if',
    'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'up',
    'about', 'into', 'through', 'during', 'this', 'that', 'these', 'those',
    'what', 'which', 'who', 'whom', 'when', 'where', 'why', 'how',
    'all', 'each', 'every', 'any', 'few', 'more', 'most', 'other',
    'very', 'just', 'than', 'too', 'same', 'as', 'out', 'also'
}


def preprocess(text):
    """
    Clean a single text string.
    Steps: lowercase → remove punctuation → remove stopwords
    """
    # Step 1: Make everything lowercase
    text = text.lower()

    # Step 2: Remove punctuation (! . , ? etc.)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Step 3: Split into individual words
    words = text.split()

    # Step 4: Remove stopwords (keep only meaningful words)
    words = [word for word in words if word not in STOPWORDS]

    # Step 5: Join words back into one string
    return ' '.join(words)


# ── Test it on one sentence ───────────────────────────────────────────────────
sample = "Safaricom customer care was so helpful and friendly!"
cleaned = preprocess(sample)

print('Original text:')
print(' ', sample)
print()
print('After preprocessing:')
print(' ', cleaned)

Original text:
  Safaricom customer care was so helpful and friendly!

After preprocessing:
  safaricom customer care helpful friendly


In [4]:
# ── Apply preprocessing to ALL rows in our dataset ────────────────────────────
df['clean_text'] = df['text'].apply(preprocess)

# Show original vs cleaned for first 5 rows
for i in range(5):
    print(f'[{df["label"][i].upper()}]')
    print(f'  Original : {df["text"][i]}')
    print(f'  Cleaned  : {df["clean_text"][i]}')
    print()

[POSITIVE]
  Original : Safaricom customer care was so helpful and friendly!
  Cleaned  : safaricom customer care helpful friendly

[POSITIVE]
  Original : M-Pesa is the best! I sent money to Kisumu in seconds.
  Cleaned  : mpesa best sent money kisumu seconds

[POSITIVE]
  Original : The nyama choma at Carnivore Restaurant is absolutely amazing!
  Cleaned  : nyama choma carnivore restaurant absolutely amazing

[POSITIVE]
  Original : KCB bank app works perfectly, very smooth experience.
  Cleaned  : kcb bank app works perfectly smooth experience

[POSITIVE]
  Original : Nairobi Expressway saved me so much time today, love it!
  Cleaned  : nairobi expressway saved much time today love



---
---
## 🔢 STEP 4 — Vectorization: Turning Words into Numbers

Machine Learning models **only understand numbers** — they cannot read words directly.

We need to convert text into a numerical form. The simplest method is called **Bag of Words (BoW)**.

### How Bag of Words works:

Imagine you have 3 sentences:

```
Sentence A: "safaricom great service"
Sentence B: "service terrible slow"
Sentence C: "safaricom slow bad"
```

The **vocabulary** (all unique words) is:
```
[safaricom, great, service, terrible, slow, bad]
```

Each sentence becomes a row of numbers — **how many times each vocabulary word appears**:

| | safaricom | great | service | terrible | slow | bad |
|--|-----------|-------|---------|----------|------|-----|
| Sentence A | 1 | 1 | 1 | 0 | 0 | 0 |
| Sentence B | 0 | 0 | 1 | 1 | 1 | 0 |
| Sentence C | 1 | 0 | 0 | 0 | 1 | 1 |

> 📌 We "ignore" the order of words (hence the name **"bag"** of words — like tossing them all in a bag). The model just counts occurrences.

Let's apply this to our dataset!

In [5]:
# ── Apply Bag of Words Vectorization ─────────────────────────────────────────
vectorizer = CountVectorizer()                  # Creates a Bag-of-Words model

X = vectorizer.fit_transform(df['clean_text'])  # Learn vocabulary + transform text to numbers
y = df['label']                                  # Labels: 'positive' or 'negative'

print('Shape of X (vectorized text):', X.shape)
print('  → Rows    =', X.shape[0], '(one row per review)')
print('  → Columns =', X.shape[1], '(one column per unique word in vocabulary)')
print()

# Show a small part of the vocabulary
vocab = vectorizer.get_feature_names_out()
print(f'Vocabulary has {len(vocab)} unique words:')
print(vocab)

Shape of X (vectorized text): (20, 111)
  → Rows    = 20 (one row per review)
  → Columns = 111 (one column per unique word in vocabulary)

Vocabulary has 111 unique words:
['absolutely' 'affordable' 'again' 'airtel' 'amazing' 'annoying' 'app'
 'atm' 'awful' 'bank' 'beautiful' 'best' 'bundles' 'bus' 'care'
 'carnivore' 'cash' 'centre' 'choma' 'coffee' 'cold' 'conductor'
 'customer' 'data' 'day' 'delivered' 'delivery' 'disappointed' 'done'
 'dropping' 'entire' 'equity' 'expensive' 'experience' 'expressway'
 'failed' 'fast' 'flights' 'food' 'friendly' 'frustrating' 'got' 'helpful'
 'highly' 'hours' 'house' 'huduma' 'interview' 'item' 'jambo' 'java' 'jet'
 'jumia' 'kbs' 'kcb' 'kept' 'kind' 'kisumu' 'late' 'loan' 'long' 'love'
 'matatu' 'meeting' 'missed' 'money' 'morning' 'mpesa' 'much' 'nairobi'
 'network' 'new' 'nightmare' 'nyama' 'overcharged' 'package' 'park'
 'perfectly' 'process' 'queues' 'recommend' 'renovation' 'restaurant'
 'road' 'rude' 'safaricom' 'saved' 'seconds' 'sent' 'serv

In [6]:
# ── See what a single review looks like as numbers ────────────────────────────
print('Original review 0:')
print(' ', df['clean_text'][0])
print()

# Convert the sparse matrix row to a dense array for display
row0 = X[0].toarray()[0]

# Only show words that appear (count > 0)
word_counts = [(word, count) for word, count in zip(vocab, row0) if count > 0]
print('As a bag-of-words (only non-zero words shown):')
for word, count in word_counts:
    print(f'   "{word}" → {int(count)}')

Original review 0:
  safaricom customer care helpful friendly

As a bag-of-words (only non-zero words shown):
   "care" → 1
   "customer" → 1
   "friendly" → 1
   "helpful" → 1
   "safaricom" → 1


---
---
## 🤖 STEP 5 — Train a Sentiment Classifier

Now that our text is turned into numbers, we can train a Machine Learning model!

We will use **Naive Bayes** — a popular, fast, and effective classifier for text.

### Why Naive Bayes for text?

It works by learning:
> *"If I see the word 'amazing', it is very likely positive. If I see 'terrible', it is very likely negative."*

It combines the probability of each word to make a final decision.

In [7]:
# ── Step 4a: Split data into train and test sets ───────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,     # 25% for testing (5 samples)
    random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')

Training samples : 15
Testing samples  : 5


In [8]:
# ── Step 4b: Train the Naive Bayes model ──────────────────────────────────────
model = MultinomialNB()          # Create the model
model.fit(X_train, y_train)      # Train it — this is where learning happens!

print('✅ Model trained successfully!')
print('   The model has learned which words are associated with positive/negative reviews.')

✅ Model trained successfully!
   The model has learned which words are associated with positive/negative reviews.


---
---
## 🔮 STEP 6 — Make Predictions

Let's see how well our model does on the test data, and also test it on **brand new sentences**!

In [9]:
# ── Predict on the test set ───────────────────────────────────────────────────
y_pred = model.predict(X_test)

# Show predictions vs actual labels
test_indices = y_test.index.tolist()
print('📊 Predictions on test set:\n')
for idx, actual, predicted in zip(test_indices, y_test, y_pred):
    status = '✅' if actual == predicted else '❌'
    print(f'{status} Review: "{df["text"][idx][:55]}..."')
    print(f'   Actual: {actual.upper():<12}  Predicted: {predicted.upper()}')
    print()

📊 Predictions on test set:

❌ Review: "Safaricom customer care was so helpful and friendly!..."
   Actual: POSITIVE      Predicted: NEGATIVE

✅ Review: "Airtel network kept dropping during my meeting, so anno..."
   Actual: NEGATIVE      Predicted: NEGATIVE

❌ Review: "The food at that restaurant was cold and tasteless, awf..."
   Actual: NEGATIVE      Predicted: POSITIVE

❌ Review: "M-Pesa is the best! I sent money to Kisumu in seconds...."
   Actual: POSITIVE      Predicted: NEGATIVE

❌ Review: "The new Uhuru Park renovation is beautiful, well done!..."
   Actual: POSITIVE      Predicted: NEGATIVE



In [10]:
# ── Evaluate the model ────────────────────────────────────────────────────────
accuracy = accuracy_score(y_test, y_pred)
print(f'🎯 Accuracy: {accuracy:.0%}')
print()
print('📋 Full Report:')
print(classification_report(y_test, y_pred))

🎯 Accuracy: 20%

📋 Full Report:
              precision    recall  f1-score   support

    negative       0.25      0.50      0.33         2
    positive       0.00      0.00      0.00         3

    accuracy                           0.20         5
   macro avg       0.12      0.25      0.17         5
weighted avg       0.10      0.20      0.13         5



In [11]:
# ── Predict on brand new sentences ────────────────────────────────────────────
# These are sentences the model has NEVER seen before!

new_comments = [
    "Safaricom bundles are great value, very happy!",
    "The matatu was so slow and the conductor was rude.",
    "Java House has the best mandazi in Nairobi!",
    "M-Pesa kept failing all morning, wasted my time.",
]

# Preprocess new text the same way we did the training data
new_cleaned = [preprocess(text) for text in new_comments]

# Vectorize using the SAME vocabulary (use transform, NOT fit_transform)
new_vectors = vectorizer.transform(new_cleaned)

# Predict!
new_predictions = model.predict(new_vectors)

print('🔮 Predictions on NEW comments:\n')
for comment, prediction in zip(new_comments, new_predictions):
    emoji = '😊' if prediction == 'positive' else '😞'
    print(f'{emoji} "{comment}"')
    print(f'   → Prediction: {prediction.upper()}')
    print()

🔮 Predictions on NEW comments:

😞 "Safaricom bundles are great value, very happy!"
   → Prediction: NEGATIVE

😞 "The matatu was so slow and the conductor was rude."
   → Prediction: NEGATIVE

😊 "Java House has the best mandazi in Nairobi!"
   → Prediction: POSITIVE

😞 "M-Pesa kept failing all morning, wasted my time."
   → Prediction: NEGATIVE



---
---
## ✏️ CLASS ACTIVITY — Write Your Own Review!

Now it is **your turn**! 

Write a comment about something in your daily life — a school canteen, a matatu route, your phone network, a restaurant, anything — and let the model predict whether it is positive or negative.

**Instructions:**
1. Change the text in `my_comment` below to your own sentence
2. Run the cell
3. See what the model predicts!
4. Try writing a sarcastic sentence — does the model get it right? 😄

In [ ]:
# ✏️ Write your own comment here!
my_comment = "The school lunch was really delicious today, I loved it!"   # ← Change this!

# Process and predict
my_cleaned  = preprocess(my_comment)
my_vector   = vectorizer.transform([my_cleaned])
my_result   = model.predict(my_vector)[0]

emoji = '😊' if my_result == 'positive' else '😞'
print(f'Your comment  : "{my_comment}"')
print(f'After cleaning: "{my_cleaned}"')
print(f'Prediction    : {my_result.upper()} {emoji}')

---
---
## 📝 Short Practice — Test Your Understanding

Answer these questions in the cell below by changing `answer_X` to your answer:

**Q1.** What does NLP stand for?  
**Q2.** Name ONE real-world application of NLP that you use.  
**Q3.** Why do we remove stopwords from text?  
**Q4.** What does vectorization mean in NLP?  
**Q5.** What does `model.fit(X_train, y_train)` actually do?

In [ ]:
answer_1 = ""  # Q1: NLP stands for ...
answer_2 = ""  # Q2: One real-world application is ...
answer_3 = ""  # Q3: We remove stopwords because ...
answer_4 = ""  # Q4: Vectorization means ...
answer_5 = ""  # Q5: model.fit() ...

print('Your answers:')
print(f'Q1: {answer_1}')
print(f'Q2: {answer_2}')
print(f'Q3: {answer_3}')
print(f'Q4: {answer_4}')
print(f'Q5: {answer_5}')

---
---
## ✅ Summary — What We Covered

| Concept | What it means |
|---------|---------------|
| **NLP** | Teaching computers to understand human language |
| **Why it's hard** | Ambiguity, sarcasm, context, slang |
| **NLP Workflow** | Collect → Label → Preprocess → Vectorize → Train → Predict |
| **Preprocessing** | Lowercase + remove punctuation + remove stopwords |
| **Bag of Words** | Count how many times each word appears per document |
| **Naive Bayes** | A classifier that learns which words are positive/negative |
| **model.fit()** | Training the model |
| **model.predict()** | Using the trained model on new text |

---

## 🚀 Next Step

Open **`nlp_workbook.ipynb`** in this same folder for:
- A real-world Kenyan dataset with 50+ labeled reviews
- TF-IDF vectorization (smarter than Bag of Words)
- Comparing multiple models
- A full end-to-end project you can put in your portfolio

---
*Practice Notebook — CCUB AI for Teens Bootcamp*